# 04쪽 · 그래프 만들고 실행하기

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## 1. State 정의

계산 결과를 담을 c, d, e를 준비합니다.

In [ ]:
from typing_extensions import TypedDict, NotRequired
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    a: int
    b: int
    c: NotRequired[int]
    d: NotRequired[int]
    e: NotRequired[int]

## 2. Node 정의

PPT의 함수를 그대로 펼쳐 적었습니다.

In [ ]:
def node1(state: State) -> dict:
    return {"c": state["a"] + state["b"]}


def node2(state: State) -> dict:
    return {"d": state["c"] * 2}


def node3(state: State) -> dict:
    return {"e": state["d"] + 5}

## 3. Node 등록과 Edge 연결

등록과 연결을 반복문으로 숨기지 않고 한 줄씩 확인합니다.

In [ ]:
builder = StateGraph(State)

builder.add_node("node1", node1)
builder.add_node("node2", node2)
builder.add_node("node3", node3)

builder.add_edge(START, "node1")
builder.add_edge("node1", "node2")
builder.add_edge("node2", "node3")
builder.add_edge("node3", END)

## 4. Compile

설계한 그래프를 실행할 수 있게 만듭니다.

In [ ]:
graph = builder.compile()

## 5. Invoke

입력은 a=10, b=20입니다.

In [ ]:
result = graph.invoke({
    "a": 10,
    "b": 20,
})

print(result)

**결과 해설**

`a=10, b=20, c=30, d=60, e=65`가 남습니다. c는 더하기, d는 두 배, e는 5 더하기 결과입니다.